In [1]:
# Basic Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder,
    FunctionTransformer
)

# Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Model
from sklearn.linear_model import LogisticRegression

# Train-Test Split and Cross Validation
from sklearn.model_selection import train_test_split, cross_val_score

# Save and Load Model
import joblib

In [2]:
# Load the complete dataset
df = pd.read_csv("data/lending_club_loan_two.csv")

print("Complete Dataset Shape:", df.shape)
display(df.head())

Complete Dataset Shape: (396030, 27)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,address
0,10000.0,36 months,11.44,329.48,B,B4,Marketing,10+ years,RENT,117000.0,...,16.0,0.0,36369.0,41.8,25.0,w,INDIVIDUAL,0.0,0.0,"0174 Michelle Gateway\r\nMendozaberg, OK 22690"
1,8000.0,36 months,11.99,265.68,B,B5,Credit analyst,4 years,MORTGAGE,65000.0,...,17.0,0.0,20131.0,53.3,27.0,f,INDIVIDUAL,3.0,0.0,"1076 Carney Fort Apt. 347\r\nLoganmouth, SD 05113"
2,15600.0,36 months,10.49,506.97,B,B3,Statistician,< 1 year,RENT,43057.0,...,13.0,0.0,11987.0,92.2,26.0,f,INDIVIDUAL,0.0,0.0,"87025 Mark Dale Apt. 269\r\nNew Sabrina, WV 05113"
3,7200.0,36 months,6.49,220.65,A,A2,Client Advocate,6 years,RENT,54000.0,...,6.0,0.0,5472.0,21.5,13.0,f,INDIVIDUAL,0.0,0.0,"823 Reid Ford\r\nDelacruzside, MA 00813"
4,24375.0,60 months,17.27,609.33,C,C5,Destiny Management Inc.,9 years,MORTGAGE,55000.0,...,13.0,0.0,24584.0,69.8,43.0,f,INDIVIDUAL,1.0,0.0,"679 Luna Roads\r\nGreggshire, VA 11650"


In [3]:
# Separate Input and Target
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (316824, 26)
Testing Data: (79206, 26)


In [4]:
# Dataset Information
df.info()

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Display columns with missing values
print("\nMissing Values Summary:")
missing_data = df.isnull().sum()

for column in df.columns:
    if missing_data[column] > 0:
        print(column, ":", missing_data[column])

<class 'pandas.DataFrame'>
RangeIndex: 396030 entries, 0 to 396029
Data columns (total 27 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   loan_amnt             396030 non-null  float64
 1   term                  396030 non-null  str    
 2   int_rate              396030 non-null  float64
 3   installment           396030 non-null  float64
 4   grade                 396030 non-null  str    
 5   sub_grade             396030 non-null  str    
 6   emp_title             373103 non-null  str    
 7   emp_length            377729 non-null  str    
 8   home_ownership        396030 non-null  str    
 9   annual_inc            396030 non-null  float64
 10  verification_status   396030 non-null  str    
 11  issue_d               396030 non-null  str    
 12  loan_status           396030 non-null  str    
 13  purpose               396030 non-null  str    
 14  title                 394274 non-null  str    
 15  dti        

In [5]:
# Separate numeric and categorical feature columns (target is not included)
numeric_cols = X_train.select_dtypes(include=np.number).columns
categorical_cols = X_train.select_dtypes(exclude=np.number).columns

# SimpleImputer on all training rows
num_imputer = SimpleImputer(strategy="median")
num_data = num_imputer.fit_transform(X_train[numeric_cols])

cat_imputer = SimpleImputer(strategy="most_frequent")
cat_data = cat_imputer.fit_transform(X_train[categorical_cols])

print("SimpleImputer completed on all training rows")
print("Numeric data shape:", num_data.shape)
print("Categorical data shape:", cat_data.shape)

# KNNImputer demonstration on a small sample to avoid excessive memory/time use.
# The main model pipeline below still uses the complete training dataset.
knn_sample = X_train[numeric_cols].sample(
    n=min(5000, len(X_train)),
    random_state=42
)

knn_imputer = KNNImputer(n_neighbors=5)
knn_data = knn_imputer.fit_transform(knn_sample)

print("\nKNNImputer demonstration completed")
print("KNN sample shape:", knn_data.shape)

SimpleImputer completed on all training rows
Numeric data shape: (316824, 12)
Categorical data shape: (316824, 14)

KNNImputer demonstration completed
KNN sample shape: (5000, 12)


In [6]:
# Ordered column
grade_order = ["A", "B", "C", "D", "E", "F", "G"]

# Ordinal Encoding
ordinal_encoder = OrdinalEncoder(
    categories=[grade_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

grade_encoded = ordinal_encoder.fit_transform(X_train[["grade"]])

print("Ordinal Encoding completed")
print("Grade values:", ordinal_encoder.categories_)

# One Hot Encoding
other_categorical_cols = [
    col for col in categorical_cols
    if col != "grade"
]

onehot_encoder = OneHotEncoder(handle_unknown="ignore")

onehot_encoded = onehot_encoder.fit_transform(
    X_train[other_categorical_cols]
)

print("\nOneHotEncoder completed")
print("Encoded categorical shape:", onehot_encoded.shape)

Ordinal Encoding completed
Grade values: [array(['A', 'B', 'C', 'D', 'E', 'F', 'G'], dtype=object)]

OneHotEncoder completed
Encoded categorical shape: (316824, 499656)


In [7]:
# Create a copy of dataset
df_features = X_train.copy()

# 1. Date Decomposition
df_features["issue_d"] = pd.to_datetime(
    df_features["issue_d"], format="%b-%Y"
)

df_features["issue_year"] = df_features["issue_d"].dt.year
df_features["issue_month"] = df_features["issue_d"].dt.month

# 2. Interaction Feature: Annual Income / Loan Amount
df_features["income_loan_ratio"] = (
    df_features["annual_inc"] / df_features["loan_amnt"]
)

# 3. Log Transformation: Annual Income
df_features["log_annual_inc"] = np.log1p(df_features["annual_inc"])

# Display new features
print("New Features Created:")
display(df_features[
    ["issue_year", "issue_month", "income_loan_ratio", "log_annual_inc"]
].head())

New Features Created:


,issue_year,issue_month,income_loan_ratio,log_annual_inc
379004,2011,12,2.162162,10.596660
24732,2015,5,2.277040,10.308986
388453,2015,7,2.285714,11.289794
178101,2012,5,3.186275,11.082158
51902,2015,4,5.714286,12.206078


In [8]:
# Function to create new features
def create_features(data):

    data = data.copy()

    # Date features
    data["issue_d"] = pd.to_datetime(
        data["issue_d"], format="%b-%Y"
    )

    data["issue_year"] = data["issue_d"].dt.year
    data["issue_month"] = data["issue_d"].dt.month

    # Income and loan ratio
    data["income_loan_ratio"] = (
        data["annual_inc"] / data["loan_amnt"]
    )

    # Log transformation
    data["log_annual_inc"] = np.log1p(data["annual_inc"])

    # Remove unnecessary columns
    data = data.drop(
        columns=["issue_d", "emp_title", "title", "address"]
    )

    return data


# Select numeric columns
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != "loan_status"]

numeric_cols = numeric_cols + [
    "issue_year",
    "issue_month",
    "income_loan_ratio",
    "log_annual_inc"
]

# Select categorical columns
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

categorical_cols = [
    col for col in categorical_cols
    if col not in ["loan_status", "issue_d", "emp_title", "title", "address", "grade"]
]

# Numeric pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Ordinal pipeline
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["A", "B", "C", "D", "E", "F", "G"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
    ("ord", ordinal_pipeline, ["grade"])
])

# Complete Pipeline
model_pipeline = Pipeline([
    ("feature_creation", FunctionTransformer(
        create_features, validate=False
    )),
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

print("Feature Engineering completed")
print("ColumnTransformer created")
print("Complete Pipeline created successfully")

Feature Engineering completed
ColumnTransformer created
Complete Pipeline created successfully


In [9]:
# 5-Fold Cross Validation
cv_scores = cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Cross Validation Scores:", cv_scores)
print("Pipeline CV Accuracy:", cv_scores.mean())

Cross Validation Scores: [0.80601278 0.80606013 0.80626529 0.80585497 0.80577299]
Pipeline CV Accuracy: 0.8059932321407531


In [10]:
# Train the complete pipeline
model_pipeline.fit(X_train, y_train)

# Save the pipeline
joblib.dump(model_pipeline, "loan_prediction_pipeline.pkl")

print("Pipeline saved successfully")

Pipeline saved successfully


In [11]:
# Load the saved pipeline
loaded_pipeline = joblib.load("loan_prediction_pipeline.pkl")

# Select 3 new raw rows
new_rows = X_test.head(3)

# Make predictions
predictions = loaded_pipeline.predict(new_rows)

print("Predictions for 3 new rows:", predictions)

Predictions for 3 new rows: ['Fully Paid' 'Fully Paid' 'Fully Paid']
